# A0 Full-Test Checkpoint Action Distribution

This notebook builds action-distribution plots from the checkpoint recorded in `outputs/full_test_eval/*.json`. It does **not** use the old mean of the last 5 logged W&B points. The full-test JSON selects the checkpoint and step; cached history supplies the nearest requested evaluation action metric at that step.

In [1]:
from pathlib import Path
import re
import sys

import pandas as pd


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
HELPER_DIR = TASK_DIR / "analysis" / "metrics" / "helpers"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import action_distribution_metrics as adm

print("Task dir:", TASK_DIR)
print("Helper dir:", HELPER_DIR)

Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Helper dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers


In [2]:
# Choose which cached run-data source to use for histories.
# Use "cpu", "gpu", or "all". Unsplit folders such as a0_hvg are kept as shared baselines.
RUN_DATA_SOURCE = "all"

# Keep this broad to include future a0_sparse16/a0_aib full-test JSONs when they exist.
FULL_TEST_CHECKPOINT_REGEX = r"^a0_"

# W&B histories do not always have an eval row exactly at checkpoint_global_step.
# "at_or_before" uses the latest eval metric at or before the checkpoint step and reports the delta.
# Other accepted values: "exact", "nearest".
STEP_MATCH_POLICY = "at_or_before"
MAX_STEP_DELTA = None  # set an integer number of env steps to reject distant cached eval points

CONFIG_FOLDERS_TO_LOAD = ["a0_hvg", "a0_sparse16", "a0_aib"]
ACTION_METRIC_SOURCE = "eval"
EVAL_METRIC_SPLIT = "test"
SAVE_FIGURES = True
SHOW_FIGURES = False

## Full-Test / Checkpoint Coverage

In [3]:
full_test = adm.load_full_test_eval_results()
pattern = re.compile(FULL_TEST_CHECKPOINT_REGEX)
a0_full_test = full_test[
    full_test["run_like"].fillna("").astype(str).map(lambda value: bool(pattern.search(value)))
].copy()

print(f"A0 full-test eval JSON rows matching {FULL_TEST_CHECKPOINT_REGEX!r}: {len(a0_full_test)}")
adm.print_full_test_eval_source_folders(a0_full_test)
display(a0_full_test[[
    "run_like",
    "checkpoint_global_step",
    "survival_percent",
    "split",
    "eval_episodes",
    "local_checkpoint_exists",
    "local_checkpoint_path",
    "local_best_test_checkpoint_path",
    "path",
]].sort_values(["run_like", "checkpoint_global_step"]))

A0 full-test eval JSON rows matching '^a0_': 6
Full-test eval JSON folders used to select checkpoint steps: 1 folder(s), 6 JSON file(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval (6 files)


,run_like,checkpoint_global_step,survival_percent,split,eval_episodes,local_checkpoint_exists,local_checkpoint_path,local_best_test_checkpoint_path,path
0,a0_hvg_01_eval_rho090_s0,14400000,82.897887,test,201,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
1,a0_hvg_01_eval_rho090_s1,14400000,96.967852,test,201,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
2,a0_hvg_01_eval_rho090_s2,14880000,92.989233,test,201,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
3,a0_hvg_04_eval_local_rho090_s0,14640000,80.667780,test,201,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
4,a0_hvg_04_eval_local_rho090_s1,14480000,98.854623,test,201,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
5,a0_hvg_04_eval_local_rho090_s2,15080000,91.628724,test,201,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...


In [4]:
checkpoint_dir = TASK_DIR / "checkpoint" / "with_obs_stats"
local_best = pd.DataFrame(
    {
        "checkpoint_path": [str(path) for path in sorted(checkpoint_dir.glob("best_test_a0_*.tar"))]
    }
)
if not local_best.empty:
    local_best["run_like"] = local_best["checkpoint_path"].map(lambda value: Path(value).stem.removeprefix("best_test_"))
    full_test_run_likes = set(a0_full_test["run_like"].dropna().astype(str))
    missing_full_test = local_best[~local_best["run_like"].isin(full_test_run_likes)].copy()
else:
    missing_full_test = local_best

print(f"Local best_test_a0 checkpoints without matching full_test_eval JSON: {len(missing_full_test)}")
display(missing_full_test)

Local best_test_a0 checkpoints without matching full_test_eval JSON: 39


,checkpoint_path,run_like
0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_00_flat_local_t020_s0
1,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_00_flat_local_t020_s1
2,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_00_flat_local_t020_s2
3,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_01_flat_local_t010_s0
4,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_01_flat_local_t010_s1
5,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_01_flat_local_t010_s2
6,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_02_flat_local_t035_s0
7,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_02_flat_local_t035_s1
8,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_02_flat_local_t035_s2
9,/Users/corentinplumet/Documents/RL_Marl2grid/T...,a0_aib_03_gate_hgreedy_sep_local_t020_s0


## Load Cached Histories

In [5]:
ctx = adm.load_action_distribution_context(
    experiment_folders=CONFIG_FOLDERS_TO_LOAD,
    action_metric_source=ACTION_METRIC_SOURCE,
    eval_split=EVAL_METRIC_SPLIT,
    run_data_source=RUN_DATA_SOURCE,
    save_figures=SAVE_FIGURES,
    show_figures=SHOW_FIGURES,
)

display(ctx["coverage"].groupby(["experiment", "family_label"], dropna=False).agg(
    expected=("expected_run_name", "count"),
    cached=("cached", "sum"),
    seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
).reset_index())

Expected configs: 54
Cached expected runs: 93 / 93
[  1/93] loading a0_aib_00_flat_local_t020_s0
[  2/93] loading a0_aib_00_flat_local_t020_s1
[  3/93] loading a0_aib_00_flat_local_t020_s2
[  4/93] loading a0_aib_01_flat_local_t010_s0
[  5/93] loading a0_aib_01_flat_local_t010_s1
[  6/93] loading a0_aib_01_flat_local_t010_s2
[  7/93] loading a0_aib_02_flat_local_t035_s0
[  8/93] loading a0_aib_02_flat_local_t035_s1
[  9/93] loading a0_aib_02_flat_local_t035_s2
[ 10/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s0
[ 11/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s1
[ 12/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s2
[ 13/93] loading a0_aib_04_flat_nonidle_t020_s0


<action_distribution_metrics:configs_and_cache>:365: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


[ 14/93] loading a0_aib_04_flat_nonidle_t020_s1
[ 15/93] loading a0_aib_04_flat_nonidle_t020_s2
[ 16/93] loading a0_aib_00_flat_local_t020_s0
[ 17/93] loading a0_aib_00_flat_local_t020_s1
[ 18/93] loading a0_aib_00_flat_local_t020_s2
[ 19/93] loading a0_aib_01_flat_local_t010_s0
[ 20/93] loading a0_aib_01_flat_local_t010_s1
[ 21/93] loading a0_aib_01_flat_local_t010_s2
[ 22/93] loading a0_aib_02_flat_local_t035_s0
[ 23/93] loading a0_aib_02_flat_local_t035_s1
[ 24/93] loading a0_aib_02_flat_local_t035_s2
[ 25/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s0
[ 26/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s1
[ 27/93] loading a0_aib_03_gate_hgreedy_sep_local_t020_s2
[ 28/93] loading a0_aib_04_flat_nonidle_t020_s0
[ 29/93] loading a0_aib_04_flat_nonidle_t020_s1
[ 30/93] loading a0_aib_04_flat_nonidle_t020_s2
[ 31/93] loading a0_hvg_00_baseline_s0
[ 32/93] loading a0_hvg_00_baseline_s1
[ 33/93] loading a0_hvg_00_baseline_s2
[ 34/93] loading a0_hvg_01_eval_rho090_s0
[ 35/93] load

,experiment,family_label,expected,cached,seeds
0,a0_aib,A0 AIB flat local target 0.10,6,6,"[0, 1, 2]"
1,a0_aib,A0 AIB flat local target 0.20,6,6,"[0, 1, 2]"
2,a0_aib,A0 AIB flat local target 0.35,6,6,"[0, 1, 2]"
3,a0_aib,A0 AIB flat non-idle target 0.20,6,6,"[0, 1, 2]"
4,a0_aib,A0 AIB gate h-greedy target 0.20,6,6,"[0, 1, 2]"
5,a0_hvg,A0 HVG baseline,3,3,"[0, 1, 2]"
6,a0_hvg,A0 HVG gate final MAP,3,3,"[0, 1, 2]"
7,a0_hvg,A0 HVG gate hierarchical,3,3,"[0, 1, 2]"
8,a0_hvg,A0 HVG global rho 0.90,3,3,"[0, 1, 2]"
9,a0_hvg,A0 HVG local rho 0.90,3,3,"[0, 1, 2]"


## Checkpoint-Aligned Action-0 Plots

In [6]:
checkpoint_action0 = adm.full_test_checkpoint_action0_fraction_by_agent(
    ctx,
    checkpoint_name_regex=FULL_TEST_CHECKPOINT_REGEX,
    step_policy=STEP_MATCH_POLICY,
    max_step_delta=MAX_STEP_DELTA,
)

# This prints the source folders and checkpoint/full-test JSON paths immediately before the figures.
adm.display_metric_output(checkpoint_action0, table_rows=80)

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/action_distribution_figures/eval_test/full_test_checkpoint_action0_fraction_by_agent_a0_hvg.html
No seed-aggregated agent-level action-0 data for a0_sparse16
No seed-aggregated agent-level action-0 data for a0_aib
Full-Test Checkpoint-Aligned Action-0 Fraction By Run And Agent
FULL_TEST_CHECKPOINT_ACTION0_LONG: (18, 50)


,run_name,run_id,experiment,comparison_group,family,family_label,seed,design,control_axis,control_value,control_label,entropy_schedule,n_steps,n_envs,rollout_action_samples,entity,agent,final_fraction_action0,std_last_fraction_action0,final_action0_count,std_last_action0_count,final_step_millions,averaged_logged_points,checkpoint_global_step,checkpoint_step_millions,selected_metric_step,selected_metric_step_millions,step_delta,step_delta_millions,step_match_policy,requested_step_policy,action_metric_column,action_metric_inverted,split_used,full_test_path,checkpoint,checkpoint_stem,run_like,local_checkpoint_path,local_checkpoint_exists,local_best_test_checkpoint_path,local_best_test_checkpoint_exists,survival_frac,survival_percent,eval_episodes,eval_action_heuristic_full_test,eval_action_rho_threshold_full_test,obs_normalization,run_label,condition_label
0,a0_hvg_01_eval_rho090_s0,MAPPO_bus14_T_0_0__I__1782725934_48269,a0_hvg,A0 HVG,a0_hvg_01_eval_rho090,A0 HVG global rho 0.90,0,eval_heuristic,variant_order,1.0,global rho 0.90,constant,2000,20,40000,agent_0,agent_0,0.849584,0.0,NaN,NaN,14.32,1,14400000,14.40,14320000.0,14.32,-80000.0,-0.08,at_or_before,at_or_before,test/explain/frac_action_0_agent_0,False,test,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_01_eval_rho090_s0,a0_hvg_01_eval_rho090_s0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,0.828979,82.897887,201,rho_threshold,0.9,checkpoint_stats,a0_hvg_01_eval_rho090_s0,A0 HVG global rho 0.90
1,a0_hvg_01_eval_rho090_s0,MAPPO_bus14_T_0_0__I__1782725934_48269,a0_hvg,A0 HVG,a0_hvg_01_eval_rho090,A0 HVG global rho 0.90,0,eval_heuristic,variant_order,1.0,global rho 0.90,constant,2000,20,40000,agent_1,agent_1,0.916268,0.0,NaN,NaN,14.32,1,14400000,14.40,14320000.0,14.32,-80000.0,-0.08,at_or_before,at_or_before,test/explain/frac_action_0_agent_1,False,test,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_01_eval_rho090_s0,a0_hvg_01_eval_rho090_s0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,0.828979,82.897887,201,rho_threshold,0.9,checkpoint_stats,a0_hvg_01_eval_rho090_s0,A0 HVG global rho 0.90
2,a0_hvg_01_eval_rho090_s0,MAPPO_bus14_T_0_0__I__1782725934_48269,a0_hvg,A0 HVG,a0_hvg_01_eval_rho090,A0 HVG global rho 0.90,0,eval_heuristic,variant_order,1.0,global rho 0.90,constant,2000,20,40000,agent_2,agent_2,0.912360,0.0,NaN,NaN,14.32,1,14400000,14.40,14320000.0,14.32,-80000.0,-0.08,at_or_before,at_or_before,test/explain/frac_action_0_agent_2,False,test,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_01_eval_rho090_s0,a0_hvg_01_eval_rho090_s0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,0.828979,82.897887,201,rho_threshold,0.9,checkpoint_stats,a0_hvg_01_eval_rho090_s0,A0 HVG global rho 0.90
3,a0_hvg_01_eval_rho090_s1,MAPPO_bus14_T_1_0__I__1782725942_9633,a0_hvg,A0 HVG,a0_hvg_01_eval_rho090,A0 HVG global rho 0.90,1,eval_heuristic,variant_order,1.0,global rho 0.90,constant,2000,20,40000,agent_0,agent_0,0.953398,0.0,NaN,NaN,14.32,1,14400000,14.40,14320000.0,14.32,-80000.0,-0.08,at_or_before,at_or_before,test/explain/frac_action_0_agent_0,False,test,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_01_eval_rho090_s1,a0_hvg_01_eval_rho090_s1,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,0.969679,96.967852,201,rho_threshold,0.9,checkpoint_stats,a0_hvg_01_eval_rho090_s1,A0 HVG global rho 0.90
4,a0_hvg_01_eval_rho090_s1,MAPPO_bus14_T_1_0__I__1782725942_9633,a0_hvg,A0 HVG,a0_hvg_01_eval_rho090,A0 HVG global rho 0.90,1,eval_heuristic,variant_order,1.0,global rho 0.90,constant,2000,20,40000,agent_1,agent_1,0.995

FULL_TEST_CHECKPOINT_ROWS: (6, 26)


,path,bad,checkpoint,checkpoint_stem,run_like,checkpoint_global_step,requested_model,requested_step,requested_step_policy,split,eval_episodes,deterministic_eval,eval_action_heuristic,eval_action_rho_threshold,obs_normalization,norm_obs_effective,obs_stats_available,survival_frac,survival_percent,intervention_gate,intervention_gate_eval_mode,created_at,local_checkpoint_path,local_checkpoint_exists,local_best_test_checkpoint_path,local_best_test_checkpoint_exists
0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,False,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_01_eval_rho090_s0,a0_hvg_01_eval_rho090_s0,14400000,None,None,exact,test,201,True,local_rho_threshold,0.9,checkpoint_stats,True,True,0.828979,82.897887,False,final_action_map,2026-06-28T16:20:19.671993+00:00,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True
1,/Users/corentinplumet/Documents/RL_Marl2grid/T...,False,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_01_eval_rho090_s1,a0_hvg_01_eval_rho090_s1,14400000,None,None,exact,test,201,True,local_rho_threshold,0.9,checkpoint_stats,True,True,0.969679,96.967852,False,final_action_map,2026-06-28T16:31:57.385701+00:00,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True
2,/Users/corentinplumet/Documents/RL_Marl2grid/T...,False,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_01_eval_rho090_s2,a0_hvg_01_eval_rho090_s2,14880000,None,None,exact,test,201,True,local_rho_threshold,0.9,checkpoint_stats,True,True,0.929892,92.989233,False,final_action_map,2026-06-28T16:27:19.884731+00:00,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True
3,/Users/corentinplumet/Documents/RL_Marl2grid/T...,False,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_04_eval_local_rho090_s0,a0_hvg_04_eval_local_rho090_s0,14640000,None,None,exact,test,201,True,local_rho_threshold,0.9,checkpoint_stats,True,True,0.806678,80.667780,False,final_action_map,2026-06-29T04:03:27.221870+00:00,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True
4,/Users/corentinplumet/Documents/RL_Marl2grid/T...,False,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_04_eval_local_rho090_s1,a0_hvg_04_eval_local_rho090_s1,14480000,None,None,exact,test,201,True,local_rho_threshold,0.9,checkpoint_stats,True,True,0.988546,98.854623,False,final_action_map,2026-06-29T04:13:55.706533+00:00,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True
5,/Users/corentinplumet/Documents/RL_Marl2grid/T...,False,/home/plumet/RL_Marl2grid/Topology_Task/checkp...,a0_hvg_04_eval_local_rho090_s2,a0_hvg_04_eval_local_rho090_s2,15080000,None,None,exact,test,201,True,local_rho_threshold,0.9,checkpoint_stats,True,True,0.916287,91.628724,False,final_action_map,2026-06-29T04:07:36.773468+00:00,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True,/Users/corentinplumet/Documents/RL_Marl2grid/T...,True


FULL_TEST_CHECKPOINT_ACTION0_MISSING: (0, 0)


""


Plot source folders used to build these action-distribution curves: 1 folder(s), 6 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (6 runs)
Full-test eval JSON folders used to select checkpoint steps: 1 folder(s), 6 JSON file(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval (6 files)
Checkpoint/full-test files used for plotted rows:
  - a0_hvg_01_eval_rho090_s0 @ step 14400000: checkpoint=/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/checkpoint/with_obs_stats/a0_hvg_01_eval_rho090_s0.tar; matching_best_test=/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/checkpoint/with_obs_stats/best_test_a0_hvg_01_eval_rho090_s0.tar; full_test_json=/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval/a0_hvg_01_eval_rho090_s0_step14400000_job64706043.json
  - a0_hvg_01_eval_rho090_s1 @ step 14400000: checkpoint=/Users/corentinplumet/Documents/RL_Marl2grid/Topology_

In [7]:
step_audit_cols = [
    "run_name",
    "agent",
    "checkpoint_global_step",
    "selected_metric_step",
    "step_delta",
    "step_match_policy",
    "action_metric_column",
    "full_test_path",
    "local_checkpoint_path",
    "local_best_test_checkpoint_path",
]
action0_rows = checkpoint_action0["tables"]["FULL_TEST_CHECKPOINT_ACTION0_LONG"]
display(action0_rows[[col for col in step_audit_cols if col in action0_rows.columns]])

,run_name,agent,checkpoint_global_step,selected_metric_step,step_delta,step_match_policy,action_metric_column,full_test_path,local_checkpoint_path,local_best_test_checkpoint_path
0,a0_hvg_01_eval_rho090_s0,agent_0,14400000,14320000.0,-80000.0,at_or_before,test/explain/frac_action_0_agent_0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
1,a0_hvg_01_eval_rho090_s0,agent_1,14400000,14320000.0,-80000.0,at_or_before,test/explain/frac_action_0_agent_1,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
2,a0_hvg_01_eval_rho090_s0,agent_2,14400000,14320000.0,-80000.0,at_or_before,test/explain/frac_action_0_agent_2,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
3,a0_hvg_01_eval_rho090_s1,agent_0,14400000,14320000.0,-80000.0,at_or_before,test/explain/frac_action_0_agent_0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
4,a0_hvg_01_eval_rho090_s1,agent_1,14400000,14320000.0,-80000.0,at_or_before,test/explain/frac_action_0_agent_1,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
5,a0_hvg_01_eval_rho090_s1,agent_2,14400000,14320000.0,-80000.0,at_or_before,test/explain/frac_action_0_agent_2,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
6,a0_hvg_01_eval_rho090_s2,agent_0,14880000,14880000.0,0.0,exact,test/explain/frac_action_0_agent_0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
7,a0_hvg_01_eval_rho090_s2,agent_1,14880000,14880000.0,0.0,exact,test/explain/frac_action_0_agent_1,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
8,a0_hvg_01_eval_rho090_s2,agent_2,14880000,14880000.0,0.0,exact,test/explain/frac_action_0_agent_2,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
9,a0_hvg_04_eval_local_rho090_s0,agent_0,14640000,14560000.0,-80000.0,at_or_before,test/explain/frac_action_0_agent_0,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...,/Users/corentinplumet/Documents/RL_Marl2grid/T...
